# 04. Baseline model: logistic regression

**Objective.** Build an interpretable baseline, and use it to test three preprocessing choices raised in earlier phases: log-transforming amounts (D11), class weights, and splines for the debt ratios (D15).

**Business question.** How well can a simple, explainable model rank applicants by risk, and are its probabilities usable as probabilities?

**Why this analysis is needed.** Every later model is judged against this one. A complex model that barely beats a logistic regression is not worth its opacity.

All fits use the training split; every score is on the **validation split**. The test split is blocked in code (`fit_and_score` refuses it).

*Educational project. Not a lending policy.*

## Reference points

A model that ranks at random has PR-AUC equal to the default rate; a model that predicts the constant rate has a fixed Brier score. Any result must be read against these.

In [1]:
import pandas as pd
from src.data import load_modelling_table
from src.evaluate import reference_scores
from src.train import *

table = load_modelling_table()
valid = table[table["split"] == "valid"]
print(table["split"].value_counts().to_string())
print(reference_scores(valid[config.TARGET]))

split
train    184506
test      61503
valid     61502
{'pr_auc_of_random_ranking': np.float64(0.08072908198107379), 'brier_of_constant_rate': np.float64(0.07421189730356685)}


**Result.** Validation has 61,502 rows at an 8.07% default rate: random PR-AUC is 0.0807 and the constant-rate Brier score is 0.0742.

**Interpretation.** ROC-AUC alone can look good on imbalanced data; PR-AUC and Brier are read against these two numbers.

## Experiment 1: three preprocessing choices on application columns only

In [2]:
application = feature_columns(table)
runs = {
    "plain": make_logistic(table, application),
    "log amounts": make_logistic(table, application, log_amounts=True),
    "balanced weights": make_logistic(table, application, class_weight="balanced"),
}
rows = {}
for name, model in runs.items():
    scores, _ = fit_and_score(model, table, application)
    rows[name] = scores
print(pd.DataFrame(rows).T.round(4).to_string())

                  roc_auc  pr_auc      ks   brier  mean_predicted  actual_rate  seconds
plain              0.7459  0.2233  0.3669  0.0687          0.0812       0.0807  21.3933
log amounts        0.7467  0.2243  0.3660  0.0687          0.0812       0.0807  25.8760
balanced weights   0.7458  0.2213  0.3657  0.2039          0.4205       0.0807  36.1403


**Result.** ROC-AUC is 0.7459 (plain), 0.7467 (log amounts) and 0.7458 (balanced). With balanced weights the mean predicted probability is 0.42 against a true rate of 0.081, and Brier rises from 0.0687 to 0.2039.

**Interpretation.**
- The log transform moves ROC-AUC by 0.0008. With about 5,000 defaulters in validation, the sampling noise in AUC is roughly 0.004, so this is not a detectable improvement. It is kept only because it is harmless and D11 said to log amounts where skew needs it.
- Balanced weights do not improve ranking (AUC 0.7458 vs 0.7459) and they wreck the probabilities. The model then reports about a 42% default rate for a portfolio where 8% default. Ranking is what weights could help; here they did not, and probability is what we need for expected loss.

**Decision.** No class weights: use the natural rate and handle the decision threshold later (Phase 7). Keep log amounts in the logistic model. See D20.

## Experiment 2: do splines help the debt ratios?

Notebook 02 found an inverted-U between credit-to-income and default. A straight line in a logistic model cannot represent that, so the test is to give the two ratios a flexible shape.

In [3]:
ratios = feature_columns(table, ratios=True)
flexible = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO"]
rows = {}
for name, model in {
    "ratios, linear": make_logistic(table, ratios, log_amounts=True),
    "ratios, splines on two": make_logistic(table, ratios, log_amounts=True, spline_columns=flexible),
}.items():
    scores, _ = fit_and_score(model, table, ratios)
    rows[name] = scores
print(pd.DataFrame(rows).T.round(4).to_string())

                        roc_auc  pr_auc      ks   brier  mean_predicted  actual_rate  seconds
ratios, linear           0.7483  0.2267  0.3706  0.0686          0.0813       0.0807  21.7824
ratios, splines on two   0.7484  0.2271  0.3701  0.0686          0.0813       0.0807  23.5195


**Result.** ROC-AUC is 0.7483 (linear) and 0.7484 (splines); PR-AUC 0.2267 and 0.2271. The ratio columns themselves add about 0.002 AUC over application-only with log amounts (0.7467).

**Interpretation.** The inverted U is real in the bins of notebook 02, but it is a small effect once the model also holds income, credit, annuity, external scores and the rest. Extra flexibility does not buy measurable ranking power for the logistic model.

**Decision.** No splines in the baseline: a gain of 0.0001 AUC is far below the noise (about 0.004), and the columns are harder to explain. The non-linearity question goes to the tree models. See D21.

## What the baseline learned

In [4]:
final = make_logistic(table, ratios, log_amounts=True)
scores, probabilities = fit_and_score(final, table, ratios)
names = final.named_steps["prepare"].get_feature_names_out()
coefficients = pd.Series(final.named_steps["model"].coef_[0], index=names).sort_values()
print("strongest protective (lower risk)")
print(coefficients.head(8).round(3).to_string())
print()
print("strongest risk-increasing")
print(coefficients.tail(8).round(3).to_string())

strongest protective (lower risk)
AMT_GOODS_PRICE                      -1.051
AMT_ANNUITY                          -0.644
ORGANIZATION_TYPE_Military           -0.526
CREDIT_ANNUITY_RATIO                 -0.524
EXT_SOURCE_3                         -0.493
CODE_GENDER_F                        -0.394
EXT_SOURCE_2                         -0.371
NAME_HOUSING_TYPE_Office apartment   -0.357

strongest risk-increasing
ORGANIZATION_TYPE_Trade: type 3        0.165
ORGANIZATION_TYPE_Industry: type 3     0.172
FLAG_DOCUMENT_3                        0.173
ORGANIZATION_TYPE_Construction         0.178
OCCUPATION_TYPE_Low-skill Laborers     0.200
ORGANIZATION_TYPE_Realtor              0.355
ORGANIZATION_TYPE_Transport: type 3    0.618
AMT_CREDIT                             2.006


**Result.** The largest coefficients are `AMT_CREDIT` (+2.01), `AMT_GOODS_PRICE` (-1.05), `AMT_ANNUITY` (-0.64) and `CREDIT_ANNUITY_RATIO` (-0.52). The external scores are smaller (`EXT_SOURCE_3` -0.49, `EXT_SOURCE_2` -0.37).

**Interpretation.** Do not read these as "bigger loans are riskier by 2.0". Credit amount, goods price and annuity are strongly correlated with each other (a loan usually finances the goods price, and the annuity is credit spread over a term), and `CREDIT_ANNUITY_RATIO` is built from two of them. Correlated inputs split their effect between them with opposite signs, so the individual coefficients are unstable even though the model's predictions are fine. The reliable reading is the group: the loan-size cluster together carries the signal. This is a real limitation of coefficient-based explanation here, and it is one reason Phase 8 explains predictions with a method that attributes effect per prediction rather than quoting these numbers. Numeric inputs are standardised, so coefficients are per one standard deviation; one-hot categories are relative to the reference category.

## Decision

The logistic baseline is the ratios feature set, log amounts, no class weights. Its validation results are the yardstick for Notebook 05.

## Next step

Notebook 05: gradient boosting and a random forest on three feature sets, an ablation of the external scores and social-circle columns, and the choice of model.